In [ ]:
import json
from typing import List, Dict

class JSONLDataProcessor:
    def __init__(self):
        pass
    
    def load_jsonl_data(self, file_path: str) -> List[Dict]:
        """Tải và xử lý dữ liệu từ file JSONL"""
        chunks = []
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f):
                    line = line.strip()
                    if not line:
                        continue
                    
                    try:
                        data = json.loads(line)
                        
                        # Kiểm tra cấu trúc dữ liệu
                        if 'text' not in data:
                            print(f"Dòng {line_num + 1}: Thiếu trường 'text'")
                            continue
                        
                        # Tạo chunk với metadata đầy đủ
                        chunk = {
                            'text': data['text'],
                            'metadata': {
                                'chunk_id': len(chunks),
                                'nam': data.get('metadata', {}).get('nam'),
                                'trieu_dai': data.get('metadata', {}).get('trieu_dai'),
                                'thuc_the': data.get('metadata', {}).get('thuc_the', []),
                                'chu_de': data.get('metadata', {}).get('chu_de'),
                                'source_line': line_num + 1
                            }
                        }
                        chunks.append(chunk)
                        
                    except json.JSONDecodeError as e:
                        print(f"Lỗi JSON dòng {line_num + 1}: {e}")
                        continue
            
            print(f"Đã tải {len(chunks)} đoạn văn bản từ file JSONL")
            return chunks
            
        except Exception as e:
            print(f"Lỗi khi đọc file: {e}")
            return []

# Thay đổi đường dẫn file JSONL của bạn tại đây
jsonl_file_path = "/kaggle/input/rag-data/final_rag_data.jsonl"

processor = JSONLDataProcessor()
chunks = processor.load_jsonl_data(jsonl_file_path)

# Hiển thị ví dụ
if chunks:
    print("\nVí dụ dữ liệu từ JSONL:")
    for i in range(min(2, len(chunks))):
        chunk = chunks[i]
        print(f"Đoạn {i+1}:")
        print(f"  Text: {chunk['text'][:150]}...")
        print(f"  Triều đại: {chunk['metadata']['trieu_dai']}")
        print(f"  Chủ đề: {chunk['metadata']['chu_de']}")
        print(f"  Thực thể: {chunk['metadata']['thuc_the']}")
        print()

In [ ]:
import os
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from typing import List, Dict
from tqdm.auto import tqdm
import math
import re

class VietnameseRAGSystem:
    def __init__(self,
                 embedding_model_name: str = "intfloat/multilingual-e5-large-instruct",
                 cross_encoder_name: str = "namdp-ptit/ViRanker",
                 cache_dir: str = "/kaggle/working"):
        
        print("Đang tải mô hình embedding tiếng Việt...")
        self.embedding_model = SentenceTransformer(embedding_model_name)
        
        print("Đang tải mô hình cross-encoder cho re-ranking...")
        self.cross_encoder = CrossEncoder(cross_encoder_name)
        
        self.index = None
        self.chunks = []
        self.metadata = []
        self.bm25_index = None
        self.normalized_texts = []
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        
        # Định nghĩa task cho multilingual-e5
        self.retrieval_task = 'Given a query about Vietnamese history, retrieve relevant historical passages that answer the query'
    
    def normalize_text(self, text):
        """Chuẩn hóa văn bản: chỉ chuyển về chữ thường, GIỮ NGUYÊN DẤU"""
        return text.lower()
    
    def create_index(self, chunks: List[Dict], use_cache: bool = True):
        """Tạo hoặc tải index từ cache"""
        faiss_cache_file = os.path.join(self.cache_dir, "rag_index.faiss")
        metadata_cache_file = os.path.join(self.cache_dir, "rag_metadata.pkl")
        
        if use_cache and os.path.exists(faiss_cache_file) and os.path.exists(metadata_cache_file):
            print("Đang tải index từ cache...")
            self.load_index(faiss_cache_file, metadata_cache_file)
            return
        
        print("Đang tạo vector embeddings...")
        self.chunks = chunks
        texts = [chunk['text'] for chunk in chunks]
        self.metadata = [chunk['metadata'] for chunk in chunks]
        
        # Tạo embeddings với batch processing để tối ưu bộ nhớ
        batch_size = 16  # Giảm batch size vì model lớn hơn
        embeddings = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Encoding batches"):
            batch = texts[i:i+batch_size]
            batch_emb = self.embedding_model.encode(batch, show_progress_bar=False, normalize_embeddings=True)
            embeddings.append(batch_emb)
        embeddings = np.vstack(embeddings)
        
        # Tạo FAISS index
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(embeddings)  # Đã normalize
        
        # Tạo BM25 index với tokenization cải tiến
        print("Đang tạo BM25 index...")
        self.normalized_texts = [self.normalize_text(text) for text in texts]
        tokenized_corpus = [re.sub(r'[.,!?;:"()]+', ' ', text).split() for text in self.normalized_texts]
        self.bm25_index = BM25Okapi(tokenized_corpus)
        
        # Lưu index vào cache
        self.save_index(faiss_cache_file, metadata_cache_file)
        
        print(f"Đã tạo và lưu index với {len(chunks)} đoạn văn bản")
        print(f"Thống kê metadata:")
        print(f" - Số triều đại duy nhất: {len(set(m.get('trieu_dai') for m in self.metadata if m.get('trieu_dai')))}")
        print(f" - Số chủ đề duy nhất: {len(set(m.get('chu_de') for m in self.metadata if m.get('chu_de')))}")
    
    def save_index(self, faiss_file: str, metadata_file: str):
        """Lưu index vào file"""
        faiss.write_index(self.index, faiss_file)
        
        index_data = {
            'chunks': self.chunks,
            'metadata': self.metadata,
            'normalized_texts': self.normalized_texts,
            'bm25_params': {
                'doc_freqs': self.bm25_index.doc_freqs,
                'doc_len': self.bm25_index.doc_len,
                'avgdl': self.bm25_index.avgdl,
                'k1': self.bm25_index.k1,
                'b': self.bm25_index.b,
                'epsilon': self.bm25_index.epsilon
            }
        }
        
        with open(metadata_file, 'wb') as f:
            pickle.dump(index_data, f)
        
        print(f"Đã lưu FAISS index: {faiss_file}")
        print(f"Đã lưu metadata: {metadata_file}")

# Khởi tạo và tạo lại index (FAISS và PKL)
print("Khởi tạo hệ thống RAG và tạo lại index...")
rag_system = VietnameseRAGSystem()
rag_system.create_index(chunks, use_cache=False)  # Tạo mới mà không dùng cache